# Lektion 17 - Erstellung lokaler KI-Agenten mit Foundry Local und Qwen

In diesem Notebook bauen Sie einen **lokalen Ingenieurassistenten**, der vollständig auf Ihrer Workstation ausgeführt wird. Es wird zu keiner Zeit eine Cloud-Inferenz verwendet. Der Assistent kann:

1. **Werkzeuge aufrufen** über Qwen-Funktionsaufrufe durch Foundry Local.
2. **Dateien auflisten und lesen** innerhalb eines sandboxed Projektverzeichnisses.
3. **Code analysieren** mit einfachen Metriken.
4. **Dokumentation durchsuchen** mit lokalem RAG (Chroma).
5. **Einen lokalen MCP-Server verwenden** (wird elegant übersprungen, wenn keiner konfiguriert ist).

Der Agent-Code sieht fast identisch zu den Cloud-Lektionen aus — nur der Client-Endpunkt wechselt von der Cloud zu `localhost`.


## Einrichtung

Bevor Sie dieses Notebook ausführen:

1. **Installieren Sie Microsoft Foundry Local** (siehe die [Dokumentation](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) für Ihr Betriebssystem).
2. **Laden Sie ein Qwen-Modell herunter und starten Sie es:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Installieren Sie die unten genannten Python-Pakete.

Alles läuft lokal. Eine Maschine mit ca. 8 GB RAM ist ein realistisches Minimum.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Verbindung zu Foundry Local herstellen

`FoundryLocalManager` lädt das Modell bei Bedarf herunter, startet den lokalen Dienst und stellt uns einen **OpenAI-kompatiblen Endpunkt** zur Verfügung. Anschließend richten wir das Standard-OpenAI-SDK darauf aus. Der API-Schlüssel ist ein lokaler Platzhalter – es sind keine Cloud-Anmeldeinformationen beteiligt.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Lokale Werkzeuge (Sandboxed-Dateioperationen)

Wir erstellen ein kleines Beispielprojekt auf der Festplatte und definieren dann Werkzeuge, die auf dieses Projektstammverzeichnis beschränkt sind. Die Sandbox-Prüfung ist selbst lokal wichtig: Ein Werkzeug, das beliebige Pfade liest, läuft mit den Berechtigungen Ihres Benutzers und kann alles berühren, worauf Sie Zugriff haben.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Lokales RAG mit Chroma

Wir betten eine kleine Menge an Dokumentationsausschnitten in eine lokale Chroma-Sammlung ein. Chroma läuft im Prozess und speichert Vektoren auf der Festplatte — kein Server, keine Cloud. Das Tool `search_docs` ruft die relevantesten Ausschnitte für eine Abfrage ab.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Die Tool-Aufruf-Schleife

Nun registrieren wir die Tools beim Modell mithilfe des OpenAI-Tools-Schemas und führen die standardmäßige Tool-Aufruf-Schleife aus – das Modell fordert ein Tool an, wir führen es lokal aus, geben das Ergebnis zurück und wiederholen dies, bis das Modell eine endgültige Antwort liefert. Qwens zuverlässiger Funktionsaufruf ermöglicht es, dass dies auf dem Gerät funktioniert.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. Lokales MCP (Optional)

MCP ist ein Transport, kein Cloud-Dienst — ein MCP-Server kann als lokaler Prozess über `stdio` laufen. Die folgende Zelle zeigt, wie Sie sich mit einem lokalen MCP-Server verbinden, wenn Sie einen konfiguriert haben (zum Beispiel einen Dateisystem-Server). Sie überspringt dies elegant, wenn `LOCAL_MCP_COMMAND` nicht gesetzt ist, sodass das Notebook dennoch vollständig ausgeführt wird.

Sicherheitshinweis: Ein lokaler MCP-Server läuft mit den Berechtigungen Ihres Benutzers. Beschränken Sie ihn auf ein Projektverzeichnis und validieren Sie seine Ausgaben, bevor Sie darauf reagieren.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Zusammenfassung

Du hast einen technischen Assistenten gebaut, der vollständig auf deinem Rechner läuft:

- **Foundry Local** stellte ein **Qwen**-Modell hinter einem OpenAI-kompatiblen Endpunkt bereit – sodass der Agenten-Code den Cloud-Lektionen entspricht.
- **Sandboxed-Tools** gaben dem Agenten Datei- und Codeanalyse-Zugriff, ohne das Projektverzeichnis zu verlassen.
- **Chroma** stellte **lokales RAG** über die Dokumentation bereit.
- **Local MCP** zeigte, wie man das MCP-Ökosystem offline wiederverwendet.

Zu keinem Zeitpunkt wurde eine Cloud-Inferenz verwendet.

### Herausforderung

Erweitere den lokalen Agenten, um:

1. **Mit mehreren MCP-Servern zu arbeiten** — verbinde einen Dateisystemserver und einen Git-Server und lasse den Agenten zwischen ihnen wählen.
2. **Lokalen Speicher zu verwenden** — speichere eine kurze Gesprächshistorie auf der Festplatte, damit sich der Assistent an frühere Dialogrunden über Notebook-Neustarts hinweg erinnert.
3. **Lokale Multi-Agenten-Orchestrierung zu unterstützen** — füge einen zweiten lokalen Agenten hinzu (z. B. einen Prüfer) und lasse die beiden an einer Aufgabe zusammenarbeiten.

In der nächsten Lektion lernst du, wie man eingesetzte KI-Agenten absichert.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
